# 02 — PPO 训练过程与诊断

这个 notebook 跑一个小型 SB3 PPO 实验，查看训练、验证选模、测试和 alpha 等权基线。合成结果只用于验证流程，不代表策略有效。

In [ ]:
import json
from datetime import datetime
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd

repo_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'src/res/rl_experimental').exists())
sys.path.insert(0, str(repo_root))

from src.res.rl_experimental.data import make_synthetic_panel  # noqa: E402
from src.res.rl_experimental.training import ExperimentConfig, train_experiment  # noqa: E402

run_name = datetime.now().strftime('notebook_training_%Y%m%d_%H%M%S')
output = repo_root / 'results/rl_experimental' / run_name
panel = make_synthetic_panel(n_steps=200, n_stocks=80, n_features=8, seed=17)
config = ExperimentConfig(
    candidate_count=20, max_holdings=10, max_weight=0.12,
    n_envs=4, rollout_steps=32, batch_size=64, update_epochs=2,
    hidden_dim=32, total_timesteps=512, eval_freq=256, seed=17,
)
config

每次 PPO iteration 先由 4 个环境各收集 32 步，共 128 条经验；随后分 minibatch 更新 actor 和 critic 两轮。验证集只用于选择 checkpoint，不参与梯度更新。

In [ ]:
metrics = train_experiment(panel, output, config)
print('parameter change L2:', round(metrics['parameter_change_l2'], 6))
print('timing:', json.dumps(metrics['timing'], indent=2))
print('validation:', json.dumps(metrics['validation'], indent=2))
print('test:', json.dumps(metrics['test'], indent=2))
print('baseline:', json.dumps(metrics['alpha_equal_weight_baseline'], indent=2))

In [ ]:
test = pd.read_csv(output / 'test_history.csv')
baseline = pd.read_csv(output / 'baseline_history.csv')
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
axes[0].plot(test['nav'], label='PPO checkpoint')
axes[0].plot(baseline['nav'], label='alpha top-N equal weight')
axes[0].set_ylabel('NAV')
axes[0].legend()
axes[1].plot(test['turnover'], alpha=.8, label='PPO turnover')
axes[1].plot(baseline['turnover'], alpha=.8, label='baseline turnover')
axes[1].set(xlabel='test step', ylabel='turnover')
axes[1].legend()
plt.tight_layout()

重点观察：参数是否更新；验证与测试是否分离；PPO 是否只是用更多换手换来收益；以及不同随机种子是否稳定。一次合成实验的领先没有统计意义。

In [ ]:
# 直接读取 TensorBoard event，观察 PPO 的 loss、KL 和 critic explained variance。
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
event_file = next((output / 'tensorboard').rglob('events.out.tfevents.*'))
events = EventAccumulator(str(event_file.parent)).Reload()
available = events.Tags()['scalars']
print('available scalar tags:', available)
wanted = [tag for tag in ['train/loss', 'train/approx_kl', 'train/value_loss', 'train/explained_variance'] if tag in available]
fig, axes = plt.subplots(len(wanted), 1, figsize=(10, 2.6 * len(wanted)), squeeze=False)
for axis, tag in zip(axes[:, 0], wanted):
    values = events.Scalars(tag)
    axis.plot([v.step for v in values], [v.value for v in values], marker='o')
    axis.set_title(tag)
plt.tight_layout()